In [1]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

ModuleNotFoundError: No module named 'torch'

In [5]:
import torch
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memory:", round(torch.cuda.get_device_properties(0).total_memory/1024**3,2), "GB")

# Large tensor size (heavy load)
size = 10000
iterations = 30

print("\nCreating tensors...")
A = torch.randn(size, size, device=device)
B = torch.randn(size, size, device=device)

# Warmup
print("Warming up GPU...")
for _ in range(5):
    C = torch.matmul(A, B)

if device.type == "cuda":
    torch.cuda.synchronize()

print("\nStarting GPU stress test...")
start = time.time()

for i in range(iterations):
    C = torch.matmul(A, B)
    if i % 5 == 0:
        print(f"Iteration {i}/{iterations}")

if device.type == "cuda":
    torch.cuda.synchronize()

end = time.time()

elapsed = end - start

print("\n========== RESULTS ==========")
print("Total time:", round(elapsed,2), "seconds")

# FLOPS estimate
operations = 2 * (size**3) * iterations
tflops = operations / elapsed / 1e12

print("Estimated Compute Power:", round(tflops,2), "TFLOPS")

if device.type == "cuda":
    print("\nGPU Memory Used:", round(torch.cuda.memory_allocated()/1024**3,2), "GB")

print("\nStress test finished 🚀")

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Memory: 8.0 GB

Creating tensors...
Warming up GPU...

Starting GPU stress test...
Iteration 0/30
Iteration 5/30
Iteration 10/30
Iteration 15/30
Iteration 20/30
Iteration 25/30

========== RESULTS ==========
Total time: 6.3 seconds
Estimated Compute Power: 9.53 TFLOPS

GPU Memory Used: 1.3 GB

Stress test finished 🚀


In [1]:
import torch
import torch.nn as nn
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Large neural network
model = nn.Sequential(
    nn.Linear(8192, 8192),
    nn.GELU(),
    nn.Linear(8192, 8192),
    nn.GELU(),
    nn.Linear(8192, 4096),
    nn.GELU(),
    nn.Linear(4096, 1000)
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

batch_size = 4096
data = torch.randn(batch_size, 8192).to(device)
target = torch.randn(batch_size, 1000).to(device)

print("\nStarting AI workload stress test...")

start = time.time()

for i in range(200):
    optimizer.zero_grad()
    
    output = model(data)
    loss = ((output - target)**2).mean()
    
    loss.backward()
    optimizer.step()
    
    if i % 20 == 0:
        print(f"Iteration {i}/200  |  Loss: {loss.item():.4f}")

if device.type == "cuda":
    torch.cuda.synchronize()

end = time.time()

print("\nTest finished")
print("Total time:", round(end-start,2), "seconds")

print("\nGPU memory used:",
      round(torch.cuda.memory_allocated()/1024**3,2), "GB")

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU

Starting AI workload stress test...
Iteration 0/200  |  Loss: 0.9997
Iteration 20/200  |  Loss: 0.9611
Iteration 40/200  |  Loss: 0.8416
Iteration 60/200  |  Loss: 0.9810
Iteration 80/200  |  Loss: 0.9424
Iteration 100/200  |  Loss: 0.9174
Iteration 120/200  |  Loss: 0.7747
Iteration 140/200  |  Loss: 0.7503
Iteration 160/200  |  Loss: 0.7886
Iteration 180/200  |  Loss: 0.7882

Test finished
Total time: 99.52 seconds

GPU memory used: 2.74 GB
